In [ ]:
import os
from glob import glob
import geopandas
import pandas
import fiona
import numpy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
from analysis_utils import *

In [ ]:
base_path = 'Z:\\jamaica\\Inputs'

In [ ]:
output_path = 'Z:\\jamaica\\Results'

In [ ]:
jamaica_crs = 3448

In [ ]:
jamaicaboundary = geopandas.read_file(os.path.join(base_path, 'jamaica.gpkg'))

In [ ]:
jamaicaboundary

In [ ]:
jamaicaboundary = jamaicaboundary.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system

In [ ]:
jamaicaboundary['area_hectares'] = 0.0001*jamaicaboundary.geometry.area # Convert area to hectares

In [ ]:
jamaica_total_area = jamaicaboundary['area_hectares'].sum()

In [ ]:
layer_list = ['2013_landuse_landcover.gpkg','hydrobasins.gpkg']
layer_name = ["Landuse", "Hydrobasins"]
layer_info = list(zip(layer_list,layer_name))
area_outputs = []

In [ ]:
for idx,(layer,layer_name) in enumerate(layer_info):
    data = geopandas.read_file(os.path.join(base_path, layer))
    data = data.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
    data['area_hectares'] = 0.0001*data.geometry.area # Convert area to hectares
    total_area = data['area_hectares'].sum()
    area_outputs.append((layer_name,total_area,100.0*total_area/jamaica_total_area))
    if layer == '2013_landuse_landcover.gpkg':
        land_use_area = data[['area_hectares', 'Classify']].groupby('Classify').sum()
        land_use_area["area_percentage"] = 100.0*land_use_area["area_hectares"]/jamaica_total_area
        land_use_area.to_csv(os.path.join(output_path, '2013_landuse_landcover_areas2.csv'))
    

In [ ]:
area_outputs

In [ ]:
fiona.listlayers("Z:\\jamaica/Inputs/hydrobasins.gpkg")

In [ ]:
hydrobasins

In [ ]:
hydrobasins.MAIN_BAS = hydrobasins.MAIN_BAS.apply(str)
hydrobasins.plot(column='MAIN_BAS', cmap='tab20', legend='true')

In [ ]:
hydrobasins_landcover = hydrobasins.overlay(landcover, how='intersection')

In [ ]:
hydrobasins_landcover

In [ ]:
hydrobasins_landcover.plot(column='MAIN_BAS', cmap='tab20', legend='true')

In [ ]:
hydrobasins_landcover[['HYBAS_ID', 'Hectares', 'Classify']].groupby(['HYBAS_ID','Classify']).sum()

In [ ]:
hydrobasins_landcover['percentage_cover'] = landcover.geometry.area